# Regression: Shared Preprocessing Audit

**Owner: Samith**  
This notebook documents the cleaning, split, encoding, scaling, outlier treatment, and feature engineering used independently in every regression algorithm notebook.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler

SEED = 42
sns.set_theme(style="whitegrid", palette="colorblind")

data_path = next(
    (parent / "data" / "health_insurance.csv" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "health_insurance.csv").exists()),
    None,
)
if data_path is None:
    raise FileNotFoundError("Run this notebook from inside the ML-301 repository.")

raw_df = pd.read_csv(data_path)
duplicate_count = int(raw_df.duplicated().sum())
df = raw_df.drop_duplicates().copy()
X = df.drop(columns="claim")
y = df["claim"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

X_train = X_train.copy()
X_test = X_test.copy()

# Age and BMI can interact in their effect on insurance cost.
for frame in (X_train, X_test):
    frame["age_bmi_interaction"] = frame["age"] * frame["bmi"]
numeric_features.append("age_bmi_interaction")

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)),
    ]), categorical_features),
])

print(f"Removed {duplicate_count:,} exact duplicates.")
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")

Removed 1,096 exact duplicates.
Training rows: 11,123; test rows: 2,781


In [2]:
missing_before = raw_df[["age", "bmi"]].isna().sum()
q1 = X_train[numeric_features[:-1]].quantile(0.25)
q3 = X_train[numeric_features[:-1]].quantile(0.75)
iqr = q3 - q1
outlier_counts = (
    (X_train[numeric_features[:-1]] < q1 - 1.5 * iqr)
    | (X_train[numeric_features[:-1]] > q3 + 1.5 * iqr)
).sum()
prepared_train = preprocessor.fit_transform(X_train)
prepared_test = preprocessor.transform(X_test)

print("Missing values before train-only imputation:")
display(missing_before.to_frame("count"))
print("IQR outlier counts in training predictors:")
display(outlier_counts.to_frame("count"))
print("Prepared shapes:", prepared_train.shape, prepared_test.shape)
assert not np.isnan(prepared_train).any()
assert not np.isnan(prepared_test).any()
assert set(X_train.index).isdisjoint(X_test.index)

Missing values before train-only imputation:


,count
age,396
bmi,956


IQR outlier counts in training predictors:


,count
age,0
weight,0
bmi,57
no_of_dependents,0
smoker,2222
bloodpressure,668
diabetes,2552
regular_ex,2567


Prepared shapes: (11123, 143) (2781, 143)


Exact duplicates are removed before splitting. Missing numeric values use training medians. Numerical predictors use robust scaling, which reduces the influence of plausible extreme patient measurements without deleting them. Categoricals use most-frequent imputation and one-hot encoding with one reference category dropped. `age_bmi_interaction` is added because age can change how BMI relates to claim cost.